# 03 — Khám phá và kiểm toán dữ liệu

EDA (*Exploratory Data Analysis*) là bước kiểm tra dữ liệu trước khi tin vào mô hình: có đủ ngày, đúng đơn vị, số liệu có hợp lý, và những giai đoạn nào biến động mạnh? Dữ liệu dừng tại **30/09/2025**. Chạy các ô từ trên xuống.

Giải thích từng phép kiểm tra và cách đọc biểu đồ: [EDA và đặc trưng](../docs/EDA_VA_DAC_TRUNG.md). Nguồn, cách tải và đơn vị từng chuỗi: [hướng dẫn dữ liệu](../data/README.md).

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Kiểm toán trước, quan sát biểu đồ sau

Hàm dưới đây đọc OHLCV, kiểm tra ngày trùng/giá trị thiếu/giá không dương, đếm phiên khối lượng bằng 0 và kiểm tra quan hệ `low ≤ open, close ≤ high`. Sau đó lưu `reports/data_audit.json` và các hình trong `figures/`.

Snapshot lưu **4.671 hàng**. Sau khi bỏ hai ngày HOSE đóng cửa 23–24/01/2018, EDA phân tích **4.669 phiên** từ **02/01/2007 đến 30/09/2025**. Ba nến bất nhất ở 17/08/2007, 10/06/2009 và 07/12/2009 được ghi trong kiểm toán và nằm trước cửa sổ thí nghiệm chung.

Giá có đơn vị **nghìn VND/cổ phiếu**, volume là **số cổ phiếu**. Lợi nhuận đơn là `close_hôm_nay / close_phiên_trước − 1`; giá trị `0.01` đọc là tăng 1%.

In [ ]:
from eda import run_eda
audit=run_eda(ROOT/'data/raw/fpt/fpt_historical_price.csv',ROOT/'figures',ROOT/'reports')

In [ ]:
for name in ['00_pipeline_overview.png','01_fpt_close.png','02_fpt_volume.png','03_daily_return_distribution.png','04_next_session_class_balance.png']:
    display(Image(filename=str(ROOT/'figures'/name)))

## Cách trình bày bốn hình

1. **Close:** quan sát giai đoạn mức giá tăng/giảm; không suy ra xu hướng sẽ tự tiếp tục.
2. **Volume:** trục `symlog` nén các số rất lớn nhưng vẫn giữ được số 0; khoảng cách trên hình không phải chênh lệch tuyến tính.
3. **Phân phối return:** số phiên tập trung quanh 0 và các đuôi biến động. Trung bình khoảng **0.0644%/phiên**, độ lệch chuẩn mẫu **1.9189%/phiên**; hai con số này mô tả toàn snapshot.
4. **Cân bằng hướng:** 2.118 chuyển tiếp tăng, 2.074 giảm, 478 đứng giá. Tổng 4.668 vì ngày cuối không có giá kế tiếp trong snapshot. Phải đọc riêng phân bố Test khi đánh giá mô hình.

Đọc nguồn và hệ quy chiếu giá trong [hướng dẫn dữ liệu](../data/README.md); không xem giá lịch sử đã chuẩn hóa như bằng chứng rằng đó là giá chưa điều chỉnh từng hiển thị trong quá khứ.